# 04 — MICM Fuel Prices | Bronze Ingestion (incremental, Auto Loader)

**Project:** RD Economic & Financial Intelligence Platform

Ingest the official MICM weekly fuel price file (Dominican open-data portal) into Bronze, preserving source values.

| Item | Value |
|---|---|
| Files | `precios-de-combustibles-2010-2026.xlsx` (authoritative, decimal precision) and `.csv` (same data rounded to whole pesos, kept for reconciliation) |
| Archive | `/Volumes/rd_economic_intelligence/bronze/micm_raw/fuel_prices/<yyyy>/<mm>/` — one folder per arrival month, files never overwritten |
| Step 1 — discover | **Auto Loader** (`binaryFile`, `trigger(availableNow=True)`, shared helper `src/shared/ingestion.py`) registers new `.xlsx` and `.csv` files in `bronze.micm_fuel_files` |
| Step 2 — capture | Each registered file not yet in Bronze is read (XLSX with openpyxl, CSV `;` latin-1) and appended as one snapshot |
| History | Each file carries the full series since 2010; Silver uses the newest snapshot of each format, older ones stay for auditing revisions |
| Target | `rd_economic_intelligence.bronze.micm_fuel_prices_raw` (one row per source row and format) |

**Source decision:** profiling showed the CSV equals the XLSX rounded to whole pesos on all 872 rows (no difference above 0.5). The XLSX keeps the published decimals, so it is the authoritative source; the CSV is ingested only to prove that reconciliation on every run.

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timezone
import sys, uuid, json, importlib
import pandas as pd

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import ingestion as ING
importlib.reload(ING)

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/micm_raw/fuel_prices/"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/micm_raw/_checkpoints/fuel_prices_files/"
FILES = f"{CATALOG}.bronze.micm_fuel_files"
TARGET = f"{CATALOG}.bronze.micm_fuel_prices_raw"
REQUIRED = ["DIA DESDE", "DIA HASTA", "MES", "AÑO", "GASOLINA PREMIUM", "GASOLINA REGULAR", "GASOIL OPTIMO", "GLP"]
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

registered = ING.register_new_files(spark, RAW_PATH, "*.{xlsx,csv}", CHECKPOINT, FILES,
    comment="Bronze: register of MICM fuel price files (XLSX and CSV) found in the raw archive by Auto Loader (one row per file).")
pending = ING.pending_files(spark, FILES, TARGET)
print(f"Auto Loader registered {registered} new file(s) · pending capture: {[f['_source_path'] for f in pending]}")

In [0]:
# Read each pending file; keep every source column, values as published (text in Bronze)
captured = {}
for f in pending:
    fmt = f["_source_file"].rsplit(".", 1)[-1].lower()
    path = ING.local_path(f["_source_path"])
    pdf = pd.read_excel(path, dtype=object) if fmt == "xlsx" else pd.read_csv(path, sep=";", encoding="latin-1", dtype=str)
    pdf.columns = [str(c).strip() for c in pdf.columns]
    absent = [c for c in REQUIRED if c not in pdf.columns]
    if absent:
        raise ValueError(f"{f['_source_path']}: layout changed, missing columns {absent}")
    cols = list(pdf.columns)
    rows = [(fmt, i, json.dumps({c: (None if pd.isna(v) else str(v)) for c, v in zip(cols, row)}, ensure_ascii=False))
            for i, row in enumerate(pdf.itertuples(index=False), start=2)]
    df = ING.lineage(spark.createDataFrame(rows, "_source_format string, _source_row_number int, record_json string"), f, BATCH_ID, INGESTION_TS)
    df.write.format("delta").mode("append").saveAsTable(TARGET)  # one atomic append per file
    captured[f["_source_path"].split("fuel_prices/")[-1]] = {"rows": len(rows), "columns": len(cols)}

if spark.catalog.tableExists(TARGET):
    spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Bronze: MICM weekly fuel prices as published (XLSX authoritative, CSV for reconciliation), one snapshot per archived file. All source columns kept in record_json.'")
snapshots = spark.table(TARGET).select("_source_path").distinct().count() if spark.catalog.tableExists(TARGET) else 0
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if captured else "NO_NEW_FILES", "registered_by_auto_loader": registered,
                                  "captured": captured, "snapshots_in_bronze": snapshots}, ensure_ascii=False))